In [89]:
import numpy as np
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix, coo_matrix, hstack
import pandas as pd

In [90]:
import pyarrow.parquet as pq
interactions_path = '/Users/alesy/kursach/data/interactions_audio_v2.parquet'
pf = pq.ParquetFile(interactions_path)
raw = next(pf.iter_batches(batch_size=600_000)).to_pandas()
raw.head(2)

,user_id,song_id,play_count,deezer_id,name,artists,genre_root,genre_sub,duration,release_date,...,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,timeSignature,valence
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,3834123571,The Cove,Jack Johnson,NaN,NaN,112,NaN,...,0.23,0.57,7,0.12,-15.35,1,0.05,123.61,4,0.38
1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBBMDR12A8C13253B,2,90270219,Entre Dos Aguas - Remastered 2014,Paco de Lucía,"latin, traditional","latin, worldwide",362,2014-01-01T00:00:00+00:00,...,0.92,0.83,4,0.08,-3.91,0,0.03,102.59,4,0.92


In [91]:
NUM_COLS = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'key','liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'timeSignature',
'valence']
CAT_COLS = ['genre_root','key','mode','timeSignature']

In [ ]:
genre_oh = raw['genre_root'].str.get_dummies(sep=', ')
key_oh = pd.get_dummies(raw['key'].astype(int), prefix='key')  # 12 колонок: key_0 ... 

        alternative  blues  classical  country  disco  edm  electro  folk  \
0                 0      0          0        0      0    0        0     0   
1                 0      0          0        0      0    0        0     0   
2                 0      0          0        0      0    0        0     0   
3                 1      0          0        0      0    0        0     1   
4                 1      0          0        0      0    0        0     0   
...             ...    ...        ...      ...    ...  ...      ...   ...   
599995            1      0          0        0      0    0        0     0   
599996            0      0          0        0      0    0        1     0   
599997            1      0          0        0      0    0        0     0   
599998            0      0          0        0      0    0        0     0   
599999            0      0          0        0      0    0        0     0   

        hip hop  holiday  ...  pop  r&b  reggae  religious  rock  ska  soul

In [93]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
raw[NUM_COLS] = scaler.fit_transform(raw[NUM_COLS])

## final df

In [103]:
df = pd.concat([raw[['user_id', 'song_id', 'play_count']], genre_oh, key_oh, raw[NUM_COLS]], axis=1)
df.head()

,user_id,song_id,play_count,alternative,blues,classical,country,disco,edm,electro,...,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,timeSignature,valence
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,0,0,0,0,0,0,0,...,-2.058018,1.835941,0.497780,-0.479151,-2.108229,0.706683,-0.313127,0.037398,0.176135,-0.494321
1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBBMDR12A8C13253B,2,0,0,0,0,0,0,0,...,1.093787,2.868997,-0.340834,-0.709361,0.977903,-1.415063,-0.543766,-0.713622,0.176135,1.693036
2,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBXHDL12A81C204C0,1,0,0,0,0,0,0,0,...,0.088864,-0.428838,-1.179448,0.556797,-0.524698,0.706683,0.494111,-0.664317,0.176135,-0.089255
3,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBYHAJ12A6701BF1D,1,1,0,0,0,0,0,0,...,-1.875304,-0.428838,0.777318,0.787008,-1.541719,0.706683,-0.543766,-0.190910,0.176135,-0.615841
4,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SODACBL12A8C13C273,1,1,0,0,0,0,0,0,...,1.093787,-0.428838,-0.340834,0.326587,0.945531,0.706683,-0.428446,0.480078,0.176135,0.153785


## разделяем interactions на test и train

In [104]:
# --- train/test split: 20% взаимодействий каждого юзера в test ---
import numpy as np

df_shuf = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_shuf['rank'] = df_shuf.groupby('user_id').cumcount()
df_shuf['cnt']  = df_shuf.groupby('user_id')['rank'].transform('max') + 1
is_test = df_shuf['rank'] < (df_shuf['cnt'] * 0.2).astype(int)

train = df_shuf[~is_test].reset_index(drop=True)
test  = df_shuf[is_test].reset_index(drop=True)
print(f'train={len(train):,}  test={len(test):,}  users_in_test={test.user_id.nunique():,}')


train=489,694  test=110,306  users_in_test=22,328


In [ ]:
# матрица train × features, взвешенная на play_count
FEATURES = [c for c in train.columns if c not in ('user_id', 'song_id', 'play_count')]
X = train[FEATURES].values                      # (n_rows, n_features)
w = train['play_count'].values.reshape(-1, 1)   # (n_rows, 1)
Xw = X * w                                      # каждую строку умножили на её play_count

# суммы по юзеру и суммы весов по юзеру
user_ids = train['user_id'].values
users_unique, inv = np.unique(user_ids, return_inverse=True)

num = np.zeros((len(users_unique), X.shape[1]), dtype=np.float64)
den = np.zeros(len(users_unique), dtype=np.float64)
np.add.at(num, inv, Xw)
np.add.at(den, inv, w.ravel())

user_matrix = num / den[:, None]                # (n_users, n_features)
user_emb = pd.DataFrame(user_matrix, index=users_unique, columns=FEATURES)


['alternative', 'blues', 'classical', 'country', 'disco', 'edm', 'electro', 'folk', 'hip hop', 'holiday', 'j-pop', 'jazz', 'kids', 'latin', 'metal', 'pop', 'r&b', 'reggae', 'religious', 'rock', 'ska', 'soul', 'soundtrack', 'spoken', 'traditional', 'key_-1', 'key_0', 'key_1', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'timeSignature', 'valence', 'rank', 'cnt']


In [ ]:
all_songs = 

In [ ]:
import math

K = 10

# подготавливаем матрицу песен один раз
af = audio_feats.drop_duplicates('song_id').reset_index(drop=True)
song_ids = af['song_id'].values
M = af[NUM].values
M_norms = np.linalg.norm(M, axis=1) + 1e-9

# словари: что юзер слушал в train (исключить из рекомендаций) и что в test (ground truth)
train_seen   = train.groupby('user_id').agg({'song_id': set, 'play_count': list}).to_dict('index')
test_truth   = test.groupby('user_id')['song_id'].apply(set).to_dict()
song_to_idx  = {s: i for i, s in enumerate(song_ids)}

def build_profile(user_id):
    rows = train[train['user_id'] == user_id]
    rows = rows[rows['song_id'].isin(song_to_idx)]
    if rows.empty:
        return None
    idx = rows['song_id'].map(song_to_idx).values
    return np.average(M[idx], axis=0, weights=rows['play_count'].values)

precisions, recalls, ndcgs = [], [], []

for u, truth in test_truth.items():
    profile = build_profile(u)
    if profile is None or not truth:
        continue

    scores = (M @ profile) / (M_norms * (np.linalg.norm(profile) + 1e-9))

    # маскируем уже прослушанное в train
    seen = train_seen.get(u, {}).get('song_id', set())
    if seen:
        mask = np.array([s in seen for s in song_ids])
        scores[mask] = -np.inf

    top = np.argpartition(-scores, K)[:K]
    top = top[np.argsort(-scores[top])]
    ranked = [song_ids[i] for i in top]

    hits = sum(1 for s in ranked if s in truth)
    precisions.append(hits / K)
    recalls.append(hits / len(truth))

    dcg  = sum(1/math.log2(i+2) for i,s in enumerate(ranked) if s in truth)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(truth), K)))
    ndcgs.append(dcg/idcg if idcg > 0 else 0.0)

print(f'Users evaluated: {len(precisions):,}')
print(f'[Content-based cosine]  Precision@{K}: {np.mean(precisions):.4f}')
print(f'[Content-based cosine]  Recall@{K}:    {np.mean(recalls):.4f}')
print(f'[Content-based cosine]  NDCG@{K}:      {np.mean(ndcgs):.4f}')


KeyboardInterrupt: 